In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics paddleocr paddlepaddle
print("\n✅ Cài xong.")


In [ ]:
import torch

# ⭐ Một số GPU Kaggle gán cho session (vd P100 đời cũ) không còn được bản torch mới nhất hỗ
#   trợ -> lỗi "CUDA error: no kernel image is available for execution on the device" dù
#   nvidia-smi vẫn thấy card. Cell này tự thử chạy 1 phép tính nhỏ trên GPU, nếu lỗi thì tự
#   động chuyển toàn bộ notebook sang chạy CPU (chậm hơn nhưng không crash) thay vì phải tự
#   sửa tay mỗi lần đổi GPU.
def pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        (torch.zeros(1) + torch.zeros(1)).cuda()
        return 0
    except RuntimeError as e:
        print("⚠️ GPU không tương thích với bản torch hiện tại, chuyển sang CPU:", e)
        return 'cpu'

DEVICE = pick_device()
print("✅ Dùng device:", DEVICE)


## 👉 ĐIỀN VÀO ĐÂY: đường dẫn `best.pt`, ảnh test, video test

Chỉ sửa các dòng trong cell ngay dưới đây, không cần đụng gì khác.

Lấy đường dẫn ở đâu: `best.pt`, ảnh, video đều phải được "Add Data" (Add Input) lên Kaggle trước
(Upload), sau đó **chạy cell đầu tiên của notebook** (`os.walk('/kaggle/input')`) — nó in ra chính
xác từng đường dẫn file đang có. Copy đúng dòng đó dán vào bên dưới.

In [ ]:
# ⬇️ 0) Đã có sẵn best.pt chưa? True = dùng luôn best.pt bên dưới, KHÔNG train (Run All sẽ tự bỏ qua phần train).
#       False = chưa có, muốn train mới từ dataset (cần điền DATA_YAML ở phần "Dataset để train" bên dưới).
DA_CO_BEST_PT = True

# ⬇️ 1) Đường dẫn tới file best.pt (chỉ cần đúng khi DA_CO_BEST_PT = True)
BEST_PLATE_WEIGHTS = '/kaggle/input/ten-dataset-cua-ban/best.pt'

# ⬇️ 2) Đường dẫn ảnh test (thêm bao nhiêu dòng cũng được)
TEST_IMAGES = [
    '/kaggle/input/ten-dataset-cua-ban/anh1.jpg',
    '/kaggle/input/ten-dataset-cua-ban/anh2.jpg',
]

# ⬇️ 3) Đường dẫn video test (chỉ cần nếu muốn chạy phần đọc biển số từ video ở cuối notebook)
VIDEO_PATH = '/kaggle/input/ten-dataset-cua-ban/video.mp4'
OUTPUT_VIDEO = '/kaggle/working/result_video.mp4'


## Dataset để train (chỉ cần nếu **chưa có** `best.pt`)

Nếu ở trên bạn đã điền đúng `BEST_PLATE_WEIGHTS`, **bỏ qua toàn bộ phần train này**, chạy thẳng
xuống phần "Detect + đọc ký tự" ở dưới.

Nếu chưa có `best.pt` và muốn train mới, cần một dataset detect biển số ở định dạng YOLOv8
(thư mục `train/images`, `train/labels`... kèm file `data.yaml`), upload qua "Add Data" giống ảnh/weights ở trên.

In [ ]:
import glob

# ⬇️ Chỉ cần điền nếu train mới (không có best.pt): đường dẫn tới data.yaml của dataset
DATA_YAML = '/kaggle/input/ten-dataset-cua-ban/data.yaml'


### Xem thử vài ảnh + nhãn trước khi train

Kiểm tra nhanh vài ảnh cùng bounding box đã gán nhãn, để chắc dataset đúng (khung bao quanh đúng
biển số) trước khi tốn thời gian train. Bỏ qua cell này nếu đã có `best.pt`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import yaml

def parse_label_line(parts, w, h):
    """Trả về (cls, x1, y1, x2, y2) dù dòng là bbox (5 số) hay polygon (>5 số)."""
    cls = int(float(parts[0]))
    vals = list(map(float, parts[1:]))
    if len(vals) == 4:
        xc, yc, bw, bh = vals
        x1 = (xc - bw / 2) * w; y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w; y2 = (yc + bh / 2) * h
    else:
        xs = [vals[i] * w for i in range(0, len(vals), 2)]
        ys = [vals[i] * h for i in range(1, len(vals), 2)]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    return cls, int(x1), int(y1), int(x2), int(y2)

if DA_CO_BEST_PT:
    print("⏭️ DA_CO_BEST_PT = True — đã có best.pt, bỏ qua preview dataset.")
else:
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    print("Số lớp:", data_cfg.get('nc'))
    print("Danh sách lớp:", data_cfg.get('names'))
    names = data_cfg.get('names')

    train_img_dir = os.path.join(os.path.dirname(DATA_YAML), 'train', 'images')
    train_lbl_dir = os.path.join(os.path.dirname(DATA_YAML), 'train', 'labels')
    sample_imgs = sorted(glob.glob(os.path.join(train_img_dir, '*')))[:6]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, img_path in zip(axes.flat, sample_imgs):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl_path = os.path.join(train_lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.split()
                    if not parts:
                        continue
                    cls, x1, y1, x2, y2 = parse_label_line(parts, w, h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    label = names[cls] if cls < len(names) else str(cls)
                    cv2.putText(img, label, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


### Train YOLOv8 detect vị trí biển số

Chỉ detect **khung bao (bounding box)** của biển số trong ảnh, không đọc ký tự — đọc ký tự nằm ở
phần OCR bên dưới. Bỏ qua cell này nếu đã có `best.pt`.

In [ ]:
from ultralytics import YOLO

if DA_CO_BEST_PT:
    print("⏭️ DA_CO_BEST_PT = True — đã có best.pt, bỏ qua train:", BEST_PLATE_WEIGHTS)
else:
    plate_model = YOLO('yolov8n.pt')

    plate_model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,           # ⭐ ảnh cả xe/khung hình lớn hơn nhiều so với detect ký tự -> cần imgsz lớn
        batch=16,
        patience=20,         # ⭐ early stop nếu không cải thiện sau 20 epoch
        device=DEVICE,
        project='/kaggle/working/yolo_plate_runs',
        name='train',
    )

    # ⭐ train() trả về kiểu khác nhau tuỳ version ultralytics (có bản trả dict,
    #   không có .save_dir) -> lấy qua plate_model.trainer.save_dir cho ổn định
    BEST_PLATE_WEIGHTS = str(plate_model.trainer.save_dir / 'weights' / 'best.pt')
    print("✅ Train xong, weights tốt nhất tại:", BEST_PLATE_WEIGHTS)


## Khởi tạo model detect (YOLO) + model đọc chữ (PaddleOCR)

Dùng chung cho cả phần test ảnh lẫn video bên dưới: YOLO detect vị trí biển số → crop → đưa
crop cho PaddleOCR đọc ra chuỗi ký tự.

In [ ]:
from ultralytics import YOLO
from paddleocr import PaddleOCR

plate_model_trained = YOLO(BEST_PLATE_WEIGHTS)

ocr = PaddleOCR(
    use_textline_orientation=True,
    lang='en',
    enable_mkldnn=False
)

def crop_plates(result, conf_thres=0.4):
    """Trả về danh sách ảnh (numpy array, BGR) đã crop từng biển số phát hiện được trong 1 result."""
    crops = []
    img = result.orig_img
    boxes = result.boxes
    if boxes is None:
        return crops
    for b in boxes:
        conf = float(b.conf[0])
        if conf < conf_thres:
            continue
        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        crops.append(img[y1:y2, x1:x2])
    return crops

def recognize_plate(plate_img):
    """Crop biển số -> chuỗi ký tự đọc được bằng PaddleOCR."""
    if plate_img is None or plate_img.size == 0:
        return ""
    result = ocr.predict(plate_img)
    texts = []
    for res in result:
        texts.extend(res['rec_texts'])
    return " ".join(texts)


## Test detect + đọc ký tự trên ảnh

Chạy model lên từng ảnh trong `TEST_IMAGES` (điền ở cell "ĐIỀN VÀO ĐÂY" trên đầu notebook): vẽ
khung phát hiện được, crop riêng vùng biển số, rồi đưa qua PaddleOCR để **đọc ra chữ** — đây là
phần nhận diện ký tự bạn cần.

In [ ]:
import cv2
import matplotlib.pyplot as plt

for TEST_IMG in TEST_IMAGES:
    result = plate_model_trained(TEST_IMG, conf=0.4, device=DEVICE, verbose=False)[0]

    annotated = result.plot()
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(TEST_IMG))
    plt.axis('off')
    plt.show()

    plate_crops = crop_plates(result)
    print(f"📋 {os.path.basename(TEST_IMG)}: phát hiện {len(plate_crops)} biển số")
    for crop in plate_crops:
        text = recognize_plate(crop)
        plt.figure(figsize=(6, 2))
        plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        plt.title(f"Đọc được: {text}")
        plt.axis('off')
        plt.show()


## Đọc ký tự biển số từ video

Cùng logic như phần ảnh ở trên nhưng chạy trên từng khung hình video, rồi ghi kết quả (khung +
chữ đọc được) ra file `OUTPUT_VIDEO`.

⭐ Kaggle **không có màn hình** để mở cửa sổ như `cv2.imshow()`/`cv2.waitKey()` chạy trên máy cá
nhân được — chạy trực tiếp sẽ bị lỗi. Nên thay vì mở cửa sổ, video ghi ra file bằng
`cv2.VideoWriter`, xem lại bằng cách tải file đó về hoặc phát ngay trong notebook bằng
`IPython.display.Video`.

In [ ]:
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy video: {VIDEO_PATH}\n"
        f"-> Kiểm tra: (1) dataset chứa video đã được Add Input chưa, "
        f"(2) tên file/đường dẫn có đúng chính tả không."
    )

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Không mở được video (có thể sai định dạng/codec): {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"🎬 Video: {frame_w}x{frame_h} @ {fps:.1f}fps, {total_frames} khung hình")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (frame_w, frame_h))

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    yolo_results = plate_model_trained(frame, device=DEVICE, verbose=False)[0]

    for box in yolo_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        plate_crop = frame[y1:y2, x1:x2]

        text = recognize_plate(plate_crop)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    writer.write(frame)
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"... đã xử lý {frame_idx}/{total_frames} khung hình")

cap.release()
writer.release()
print(f"✅ Xong, video kết quả lưu tại: {OUTPUT_VIDEO}")


In [ ]:
from IPython.display import Video

Video(OUTPUT_VIDEO, embed=True, width=640)
